# 07 — Pipeline d’ingestion hors ligne et cache de déploiement

Ce notebook produit les artefacts immuables utilisés par l’application. Docling est l’extracteur principal ciblé ; la baseline vérifiée PyMuPDF reste disponible lorsque Docling n’est pas installé ou lorsqu’un contrôle indépendant des coordonnées est nécessaire.

In [ ]:
from pathlib import Path
import sys

root_hint = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(root_hint / 'notebooks'))
from helpers import bootstrap, display_table, module_available
ROOT = bootstrap()
from ingestion.pipeline import ingest_pdf

PDF = ROOT / 'data' / 'raw' / 'foyer-groupe-qrt-public-2025.pdf'
OUTPUT_ROOT = ROOT / 'data' / 'processed'
RUN_INGESTION = PDF.exists()
{'pdf': str(PDF), 'pdf_available': RUN_INGESTION, 'docling_available': module_available('docling'), 'output': str(OUTPUT_ROOT)}

In [ ]:
result = ingest_pdf(
    path=PDF,
    document_id='foyer_group_qrt_2025',
    title='QRT public 2025 — Groupe Foyer',
    entity='Groupe Foyer',
    period='2025',
    source_url='https://www.foyer.lu/fr/mydoc/WebSites-Documentsgroupe-376',
    output_root=OUTPUT_ROOT,
) if RUN_INGESTION else None
result.manifest if result else 'QRT absent : téléchargement local requis pour exécuter l’ingestion.'

In [ ]:
cells = [cell for table in result.tables for cell in table.cells] if result else []
if result:
    assert len(result.pages) == 10
    assert {cell.row_code for cell in cells} == {'R0660', 'R0680', 'R0690'}
    assert {cell.column_code for cell in cells} == {'C0010'}
    assert all(cell.provenance.page == 7 and cell.provenance.bbox for cell in cells)
display_table([
    {
        'row': cell.row_code, 'column': cell.column_code,
        'raw': cell.raw_value, 'normalized': cell.normalized_value,
        'unit': cell.unit, 'page': cell.provenance.page,
    }
    for cell in cells
])

In [ ]:
expected = {
    'manifest.json', 'pages.jsonl', 'sections.json', 'tables.jsonl',
    'facts.jsonl', 'figures.jsonl', 'chunks.jsonl', 'references.json',
}
produced = {path.name for path in result.output_path().iterdir() if path.is_file()} if result else set()
if result:
    assert expected <= produced
{'cached_files': sorted(produced), 'chunks': len(result.chunks) if result else 0, 'warnings': result.manifest.warnings if result else []}

### Frontière de confiance

Les artefacts sont un cache d’ingestion hors ligne. Ils doivent être revus avant promotion dans `backend/app/data/`. Le service déployé ne télécharge aucun modèle et ne réanalyse aucun PDF.